In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
import scipy
import sklearn
cwd = Path('.')

In [2]:
data_path = cwd / 'data' / '2025_LoL_esports_match_data_from_OraclesElixir_imputated.csv'
data = pd.read_csv(data_path,index_col=0)
data_len = int(len(data) * 10 / 12)

C:\Users\dwarf\AppData\Local\Temp\ipykernel_5580\4154608675.py:2: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(data_path,index_col=0)


In [3]:
data = data.drop(data[data["position"] == "team"].index) 

In [ ]:
pick_set = []
ban_set = []
for i in range(0,len(data),5):
    data_game = data.iloc[i:i+5]
    pick_set_game = []
    ban_set_game = []
    champs_in_game = data_game["champion"]
    if (data_game["ban1"].hasnans == False): #Some bans are missing for some games, these are assumed to be as a result of a team using all their time and missing the opputunity to ban and as such the game can sstill be kept
        ban_set_game.append(data_game["ban1"])
    if (data_game["ban2"].hasnans == False):
        ban_set_game.append(data_game["ban2"])
    if (data_game["ban3"].hasnans == False):
        ban_set_game.append(data_game["ban3"])
    if (data_game["ban4"].hasnans == False):
        ban_set_game.append(data_game["ban4"])
    if (data_game["ban5"].hasnans == False):
        ban_set_game.append(data_game["ban5"])
    for champ in champs_in_game:
        pick_set_game.append(champ)
    pick_set.append(pick_set_game)
    ban_set.append(np.unique(ban_set_game))






In [7]:
pick_set = []
ban_set = []
for i in range(0,len(data),5):
    data_game = data.iloc[i:i+5]
    pick_set_game = []
    ban_set_game = []
    champs_in_game = data_game["champion"]
    if (data_game["ban1"].hasnans == False): #Some bans are missing for some games, these are assumed to be as a result of a team using all their time and missing the opputunity to ban and as such the game can sstill be kept
        ban_set_game.append(data_game.iloc[0]["ban1"]+"_banned")
    if (data_game["ban2"].hasnans == False):
        ban_set_game.append(data_game.iloc[0]["ban2"]+"_banned")
    if (data_game["ban3"].hasnans == False):
        ban_set_game.append(data_game.iloc[0]["ban3"]+"_banned")
    if (data_game["ban4"].hasnans == False):
        ban_set_game.append(data_game.iloc[0]["ban4"]+"_banned")
    if (data_game["ban5"].hasnans == False):
        ban_set_game.append(data_game.iloc[0]["ban5"]+"_banned")
    for champ in champs_in_game:
        champ_name = champ + "_picked"
        pick_set_game.append(champ_name)
    pick_set.append(pick_set_game)
    ban_set.append(np.unique(ban_set_game))






In [10]:
from mlxtend.preprocessing import TransactionEncoder
te = TransactionEncoder()
te_ary = te.fit(pick_set).transform(pick_set)
df_ap = pd.DataFrame(te_ary, columns=te.columns_)

In [82]:
df_ap

,Aatrox,Ahri,Akali,Akshan,Alistar,Ambessa,Amumu,Anivia,Annie,Aphelios,...,Yunara,Yuumi,Zaahen,Zac,Zed,Zeri,Ziggs,Zilean,Zoe,Zyra
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,True,False,False,False,True,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20101,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
20102,False,False,False,False,False,True,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
20103,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
20104,True,True,False,False,False,False,False,False,False,False,...,True,False,False,False,False,False,False,False,False,False


In [97]:
from mlxtend.frequent_patterns import apriori

frequent_itemsets = apriori(df_ap, min_support=0.01,use_colnames=True)

We need to use a very low support to get sets of a size greater than 1 since there are only ever 5 champions and very many combinations of them.

In [98]:
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(lambda x: len(x))

In [102]:
freq_greater_1 = frequent_itemsets[(frequent_itemsets['length'])>1]

In [103]:
freq_greater_1.sort_values(by=["support"])

,support,itemsets,length
88,0.010047,"(Ahri, Ambessa)",2
170,0.010047,"(Nautilus, K'Sante)",2
196,0.010096,"(Orianna, Rakan)",2
232,0.010146,"(Varus, Viktor)",2
195,0.010146,"(Nocturne, Orianna)",2
...,...,...,...
208,0.020790,"(Rell, Rumble)",2
153,0.022481,"(Ezreal, Leona)",2
212,0.022879,"(Rell, Varus)",2
145,0.024072,"(Rell, Corki)",2


Xayah and Rakan are a typical combo so seeing them with the highest support is good

In [11]:
from mlxtend.frequent_patterns import fpgrowth

In [106]:
frequent_itemsets_fpg = fpgrowth(df_ap, min_support=0.01,use_colnames=True)

In [107]:
frequent_itemsets_fpg['length'] = frequent_itemsets_fpg['itemsets'].apply(lambda x: len(x))
freq_greater_1_fpg = frequent_itemsets_fpg[(frequent_itemsets_fpg['length'])>1]
freq_greater_1_fpg.sort_values(by=["support"])

,support,itemsets,length
123,0.010047,"(Nautilus, K'Sante)",2
184,0.010047,"(Ahri, Ambessa)",2
100,0.010096,"(Orianna, Rakan)",2
188,0.010146,"(Nocturne, Orianna)",2
166,0.010146,"(Varus, Viktor)",2
...,...,...,...
192,0.020790,"(Rell, Rumble)",2
89,0.022481,"(Ezreal, Leona)",2
96,0.022879,"(Rell, Varus)",2
126,0.024072,"(Rell, Corki)",2


In [12]:
pick_ban_set = [] # First is champ selected, second is ban
for i in range(len(pick_set)):
    for champ in pick_set[i]:
        for ban in ban_set[i]:
            pick_ban_set.append([champ,ban])

In [13]:
te = TransactionEncoder()
te_ary = te.fit(pick_ban_set).transform(pick_ban_set)
df_ap_pb = pd.DataFrame(te_ary, columns=te.columns_)

In [15]:
frequent_itemsets_pb = fpgrowth(df_ap_pb, min_support=0.001,use_colnames=True)
frequent_itemsets_pb['length'] = frequent_itemsets_pb['itemsets'].apply(lambda x: len(x))
freq_greater_1_pb = frequent_itemsets_pb[(frequent_itemsets_pb['length'])>1]
freq_greater_1_pb.sort_values(by=["support"])

,support,itemsets,length
206,0.001005,"(Varus_banned, Alistar_picked)",2
209,0.001021,"(Poppy_banned, Ambessa_picked)",2
205,0.001023,"(Corki_picked, Varus_banned)",2
207,0.001045,"(Varus_banned, Taliyah_picked)",2
212,0.001079,"(Gwen_banned, Xin Zhao_picked)",2
204,0.001099,"(Rell_picked, Gwen_banned)",2
208,0.001109,"(Poppy_banned, Rell_picked)",2
213,0.001121,"(Gwen_banned, Sion_picked)",2
202,0.001131,"(Vi_banned, Rell_picked)",2
211,0.001139,"(Varus_banned, Xin Zhao_picked)",2


In [20]:
frequent_itemsets_pb.iloc[0]["itemsets"]==set(["Corki_banned"])

True

In [25]:
print(frequent_itemsets_pb[frequent_itemsets_pb["itemsets"]==set(["Varus_banned"])]["support"])
frequent_itemsets_pb[frequent_itemsets_pb["itemsets"]==set(["Rell_picked"])]["support"]

86    0.03797
Name: support, dtype: float64


25    0.030006
Name: support, dtype: float64

In [26]:
0.03797 * 0.030006

0.0011393278199999999

In [30]:
0.001175/0.0011393278199999999

1.031309847239577

In [27]:
print(frequent_itemsets_pb[frequent_itemsets_pb["itemsets"]==set(["Poppy_banned"])]["support"])
frequent_itemsets_pb[frequent_itemsets_pb["itemsets"]==set(["Ambessa_picked"])]["support"]

55    0.029079
Name: support, dtype: float64


67    0.024005
Name: support, dtype: float64

In [29]:
0.001021/(0.029079*0.024005)

1.4626639728149646

Varus is not really a good counter for rumble according to https://u.gg/lol/champions/rumble/counter, though we are seeing Varus appear high so he may just be a high priority ban

In [113]:
te = TransactionEncoder()
te_ary = te.fit(ban_set).transform(ban_set)
df_ap_ban = pd.DataFrame(te_ary, columns=te.columns_)

In [116]:
frequent_itemsets_fpg = fpgrowth(df_ap_ban, min_support=0.1,use_colnames=True)
frequent_itemsets_fpg['length'] = frequent_itemsets_fpg['itemsets'].apply(lambda x: len(x))
#freq_greater_1_fpg = frequent_itemsets_fpg[(frequent_itemsets_fpg['length'])>1]
frequent_itemsets_fpg.sort_values(by=["support"])

,support,itemsets,length
1,0.103800,(Skarner),1
12,0.103850,(Neeko),1
2,0.104049,(Yone),1
4,0.109271,(Ambessa),1
10,0.115289,(Taliyah),1
7,0.117030,(Jayce),1
3,0.133045,(Kalista),1
6,0.144783,(Poppy),1
13,0.145031,(Pantheon),1
11,0.154780,(Azir),1


As expected based on above, Varus is just a high priority ban

Can look more into counter picks (IE make each entry the laners for a given game on opposing sides), seperate by whether the team won or not, add a league or patch to see if a champ is really popular in a certain patch or league (could also seperate those into their own subdatasets). Look at the class roles (Tank, Mage etc), llok more clearly at pick ban, IE make the variables "Varus_picked" and "Varus_banned". Normalize by popularity see above